# ChemRover experiments

This notebook runs **the same pipeline as `uv run chemrover`** (`chemrover.cli.run`), but lets you change the defaults and compare runs side by side.

Changes use the same override strings as the command line:

| Change | Override |
|---|---|
| Model | `"model=rf"` (xgb, rf, svm, knn) |
| Dataset | `"data.subset=2"` (1 Byadi, 2 Griffiths, 3 PCCP, 4 combined) |
| Drop solvent (structure only) | `"data.features=[lambda,SMILES]"` |
| Split seed | `"data.seed=7"` |
| Tuning (test) / benchmark (val) fraction | `"data.test_size=0.1"`, `"data.val_size=0.2"` |
| Descriptor coverage filter | `"data.coverage_thresh=0.95"` |
| Tuning budget | `"trainer.n_trials=50"` |
| A fixed model param (only used if not tuned) | `"model.max_depth=4"` |

Every run is saved to `outputs/notebook/<date>/<time>_<label>/` in the same layout as a CLI run (model, parity plot, `metrics.json`, `predictions.csv`, `.hydra/config.yaml`), so **section 6** can compare everything you've ever run, from the notebook or the CLI.

> Kernel: select the project's `.venv` interpreter (`uv sync` installs `ipykernel`).

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import optuna
import pandas as pd
from omegaconf import OmegaConf

from chemrover.experiment import compose_cfg, run_experiment, run_grid, product, parity_grid, load_runs
from chemrover.util import setup_logging

setup_logging()                                            # ChemRover messages to the console
optuna.logging.set_verbosity(optuna.logging.WARNING)       # hide Optuna's per-trial messages
pd.set_option("display.precision", 3)

## 1. Defaults
The resolved config every run starts from. Edit the YAML files in `config/` to change the defaults permanently, or pass overrides below to change them for one run.

In [ ]:
print(OmegaConf.to_yaml(compose_cfg(), resolve=True))

Overrides applied to **every** run in this notebook. Keep the trial count low while exploring, and raise it for final numbers.
Optuna studies are cached per data + model config, so re-running an identical setup reuses its trials instead of starting again.

In [ ]:
QUICK = ["trainer.n_trials=20", "trainer.show_progress_bar=false"]

## 2. Baseline

In [ ]:
baseline = run_experiment("baseline", QUICK)
baseline.metrics

## 3. One change at a time
Change a single setting and compare it with the baseline.

In [ ]:
compare_summary, compare = run_grid(
    {"with solvent": [], "no solvent": ["data.features=[lambda,SMILES]"]}, common=QUICK,
)
display(compare_summary)
parity_grid(compare, title="Effect of including solvent")
plt.show()

## 4. Grid of settings
`product()` builds every combination of several axes. Each axis maps a readable label to one override, or a list of overrides.

> Note: rows missing a value in any kept column are dropped, so "with solvent" can train on fewer molecules than "no solvent" (e.g. Byadi has rows with no solvent). Check `n_train` before comparing.

In [ ]:
variants = product(
    dataset={
        "Byadi": "data.subset=1",
        "Griffiths": "data.subset=2",
        "PCCP": "data.subset=3",
        "Combined": "data.subset=4",
    },
    solvent={
        "with solvent": [],
        "no solvent": "data.features=[lambda,SMILES]",
    },
)
summary, results = run_grid(variants, common=QUICK)   # index: (dataset, solvent)
summary

In [ ]:
# one metric as a dataset x solvent table
summary["r2"].unstack()

In [ ]:
ax = summary["rmse"].unstack().plot.bar(
    rot=0, ylabel="Val RMSE / nm", title="Val RMSE by dataset and solvent", figsize=(8, 4),
)
plt.show()

In [ ]:
parity_grid(results, ncols=4, show_residuals=False)
plt.show()

## 4b. Compare models
`model=<name>` switches the whole model config group (`config/model/<name>.yaml`). Each model is tuned with its own Optuna search space (`PARAM_LIMS` in `src/chemrover/models/<name>.py`).
Add a dataset or solvent axis to `product()` to see whether the best model depends on the data.

In [ ]:
models = product(model={
    "Mean (baseline)": "model=dummy", "PLS": "model=pls", "KNN": "model=knn", "SVM": "model=svm",
    "Random forest": "model=rf", "XGBoost": "model=xgb", "GP": "model=gp", "MLP": "model=mlp",
})
model_summary, model_results = run_grid(models, common=QUICK)
model_summary.sort_values("rmse")

In [ ]:
parity_grid(model_results, ncols=4, show_residuals=False)
plt.show()

## 5. How much is just split noise?
With datasets this small, the random data split alone moves the metrics. Repeat one setting over several seeds: **differences between settings that are smaller than this spread aren't meaningful.**

In [ ]:
seeds = {f"seed {s}": [f"data.seed={s}"] for s in range(5)}
seed_summary, seed_results = run_grid(seeds, common=QUICK)
seed_summary[["rmse", "mae", "r2"]].agg(["mean", "std"])

## 5b. Learning curve: does solvent help more as the data grows?
XGBoost on the combined dataset, trained on increasing fractions of the training split, **with and without solvent**.
- `data.train_fraction` shrinks only the training split. The test (tuning) and val (benchmark) sets stay fixed, so every point is scored on the same molecules.
- Smaller fractions are nested inside larger ones (the 5% sample is inside the 10% sample), and `data.required` means both solvent variants use identical rows. The only difference between the two lines is the solvent column.
- Each point is tuned separately (each fraction has its own Optuna study) and repeated over 3 split seeds, drawn as mean ± 1 sd.

**Reading it:** the vertical gap between the lines is what solvent adds at that data size. A line still falling at the right-hand edge means more data would still help; a flat line means the model or representation (or measurement noise) is the limit.

In [ ]:
# 7 training sizes x with/without solvent x 3 split seeds = 42 runs (about 10 min with QUICK)
fractions = [0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0]
curve_variants = product(
    solvent={"with solvent": [], "no solvent": "data.features=[lambda,SMILES]"},
    fraction={f"{f:.2f}": f"data.train_fraction={f}" for f in fractions},
    seed={str(s): f"data.seed={s}" for s in range(3)},
)
curve, curve_results = run_grid(curve_variants, common=QUICK + ["model=xgb", "data.subset=4"])
curve.index.names = ["solvent", "fraction", "seed"]

# mean and spread over the seeds at each training-set size
curve_stats = curve.groupby(["solvent", "n_train"])[["rmse", "mae", "r2"]].agg(["mean", "std"])
curve_stats

In [ ]:
import matplotlib as mpl
from chemrover.plotting.pred_vs_measured import _RC   # same light style as the parity plots

colors = {"with solvent": "#2a78d6", "no solvent": "#eb6834"}    # categorical slots 1 and 2
styles = {"with solvent": ("o", "-"), "no solvent": ("s", "--")}  # marker + dash, so identity isn't colour alone


def plot_curve(ax, metric, ylabel, legend_loc):
    """Mean ± 1 sd of `metric` against training-set size, one line per solvent setting."""
    for solvent, grp in curve_stats.groupby(level="solvent"):
        n = grp.index.get_level_values("n_train")
        mean, std = grp[(metric, "mean")], grp[(metric, "std")]
        marker, ls = styles[solvent]
        ax.fill_between(n, mean - std, mean + std, color=colors[solvent], alpha=0.15, lw=0)
        ax.plot(n, mean, ls=ls, lw=2, marker=marker, ms=8, mec="white", mew=2,
                color=colors[solvent], label=solvent)
    ax.set(xlabel="Training molecules", ylabel=ylabel)
    ax.legend(loc=legend_loc)
    ax.grid(True, color="#e3e6e8", lw=0.7)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)


with mpl.rc_context(_RC):
    fig, (ax_rmse, ax_r2) = plt.subplots(1, 2, figsize=(12, 4.5))
    plot_curve(ax_rmse, "rmse", "Val RMSE / nm", legend_loc="upper right")
    plot_curve(ax_r2, "r2", "Val R²", legend_loc="lower right")
    fig.suptitle("XGBoost learning curve, combined dataset (mean ± 1 sd over 3 splits)",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    plt.show()

In [ ]:
# what solvent adds at each size: positive = including solvent lowers the val RMSE
gap = curve_stats[("rmse", "mean")].unstack("solvent")
gap["solvent benefit / nm"] = gap["no solvent"] - gap["with solvent"]
gap

## 5c. Learning curve by model
The same setup as 5b (combined dataset, fixed test and val sets, nested training subsets, 3 split seeds), with solvent included, comparing the models instead of the solvent setting. The mean baseline is left out: it can't improve with more data.

- **Size effect:** how each model's error falls as the training set grows, and which models do best with little data.
- **Split variation:** the spread over the 3 seeds at each size, i.e. how much a model's score depends on which molecules land in each split.

7 models × 7 sizes × 3 seeds = 147 runs, which takes roughly an hour with `QUICK` (random forest and GP dominate). To shorten it, trim `lc_models`, `lc_fractions` or the seeds.

In [ ]:
lc_models = {
    "PLS": "model=pls", "KNN": "model=knn", "SVM": "model=svm", "Random forest": "model=rf",
    "XGBoost": "model=xgb", "GP": "model=gp", "MLP": "model=mlp",
}
lc_fractions = [0.05, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0]
model_curve, _ = run_grid(       # fitted models aren't kept: 147 of them would use a lot of memory
    product(
        model=lc_models,
        fraction={f"{f:.2f}": f"data.train_fraction={f}" for f in lc_fractions},
        seed={str(s): f"data.seed={s}" for s in range(3)},
    ),
    common=QUICK + ["data.subset=4"],
)
model_curve.index.names = ["model", "fraction", "seed"]

# mean and spread over the seeds, per model and training-set size (models kept in lc_models order)
model_curve_stats = model_curve.groupby(["model", "n_train"], sort=False)[["rmse", "mae", "r2"]].agg(["mean", "std"])
model_curve_stats

In [ ]:
import matplotlib as mpl
from chemrover.plotting.pred_vs_measured import _RC   # same light style as the parity plots

# categorical slots 1-7 in fixed order, plus a distinct marker per model so identity isn't colour alone
model_colors = dict(zip(lc_models, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]))
model_markers = dict(zip(lc_models, "osD^vPX"))
by_model = list(model_curve_stats.groupby(level="model", sort=False))

# Filter
models_to_plot = ["KNN", "SVM", "Random forest", "XGBoost", "GP"]
temp = []
for model, grp in by_model:
    if model in models_to_plot:
        temp.append((model, grp))
by_model = temp

def style_axes(ax, ylabel):
    ax.set(xlabel="Training molecules", ylabel=ylabel)
    ax.grid(True, color="#e3e6e8", lw=0.7)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)


def plot_line(ax, model, grp, column, **kwargs):
    ax.plot(grp.index.get_level_values("n_train"), grp[column], lw=2, ms=7, mec="white", mew=1.5,
            marker=model_markers[model], color=model_colors[model], **kwargs)


with mpl.rc_context(_RC):
    # 1) all models together: dataset-size effect (means) and split variation (sd over seeds)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    panels = [(("rmse", "mean"), "Val RMSE / nm (mean)", (0, 50)),
              (("r2", "mean"),   "Val R² (mean)",        (0, 1)),
              (("rmse", "std"),  "Val RMSE / nm (sd over splits)", (0, 10))]
    for ax, (column, ylabel, ylim) in zip(axes, panels):
        for model, grp in by_model:
            plot_line(ax, model, grp, column, label=model)
        ax.set_ylim(*ylim)
        style_axes(ax, ylabel)
    axes[-1].legend(loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.suptitle("Learning curves by model, combined dataset (3 splits)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    plt.show()

    # 2) one panel per model, shared axes: mean ± 1 sd band shows the split variation at each size
    fig, axes = plt.subplots(2, 3, figsize=(16, 7), sharex=True, sharey=True)
    for ax, (model, grp) in zip(axes.flat, by_model):
        n, mean, std = grp.index.get_level_values("n_train"), grp[("rmse", "mean")], grp[("rmse", "std")]
        ax.fill_between(n, mean - std, mean + std, color=model_colors[model], alpha=0.2, lw=0)
        plot_line(ax, model, grp, ("rmse", "mean"))
        ax.set_title(model, loc="left", fontsize=11)
        ax.set_ylim(0, 50)
        style_axes(ax, "Val RMSE / nm")
        ax.label_outer()
    axes[1, 2].set_visible(False)                           # 7 models in a 2 x 4 grid
    axes[0, 2].tick_params(labelbottom=True)                # so the last column keeps its x axis
    axes[0, 2].set_xlabel("Training molecules")
    fig.suptitle("Val RMSE by model: mean ± 1 sd over 3 splits", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    plt.show()

In [ ]:
# each model at the smallest and the full training size, best first at full size
sizes = sorted(model_curve["n_train"].unique())
ends = pd.concat({f"{n} molecules": model_curve_stats.xs(n, level="n_train")[["rmse", "r2"]]
                  for n in (sizes[0], sizes[-1])}, axis=1)
ends.sort_values((f"{sizes[-1]} molecules", "rmse", "mean"))

## 6. Run history
Every saved run with a `metrics.json`, from the notebook and the CLI, newest first.

In [ ]:
runs = load_runs()
runs.head(20)

In [ ]:
# e.g. every run that dropped solvent, best first
runs[runs["overrides"].str.contains("features=[lambda,SMILES]", regex=False)].sort_values("r2", ascending=False)

## 7. Inspect one run
`RunResult.predictions` holds the val-set predictions with any categorical context (e.g. solvent), which is useful for seeing where the errors come from.

In [ ]:
res = results[("Combined", "with solvent")]
errors = res.predictions.assign(abs_err=res.predictions["residual"].abs())
errors.groupby("solvent", observed=True)["abs_err"].agg(["count", "mean", "max"]).sort_values("count", ascending=False)

## 8. Your experiment
Copy this cell to test anything else.

In [ ]:
my_variants = {
    "coverage 0.90": ["data.coverage_thresh=0.90"],
    "coverage 0.99": ["data.coverage_thresh=0.99"],
}
my_summary, my_results = run_grid(my_variants, common=QUICK)
my_summary